# Bloom — Instruction tuning pilot

Experimental response-only supervised fine-tuning from the published base checkpoint. Use original generated arithmetic and basic factual/formatting examples, or add a separately reviewed local JSONL corpus of prompt/response pairs. Keep this candidate separate from the base model. This small pilot tests the format and training mechanics, not broad chat ability. CUDA required; no automatic publication.

In [ ]:
# 1. Clone Centauri Bloom (main) and install dependencies
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = Path('/kaggle/working').is_dir()
WORKSPACE = Path('/content') if IN_COLAB else (Path('/kaggle/working') if IN_KAGGLE else Path.cwd())
if Path('pyproject.toml').is_file() and Path('bloom/model.py').is_file():
    REPO = Path.cwd()
else:
    REPO = WORKSPACE / 'centauri-bloom'
    if not (REPO / 'pyproject.toml').is_file():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/InterstellarNerd8760/centauri-bloom.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers', 'datasets'], check=True)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
# 2. Configure run & choose persistent output
import json, torch
from pathlib import Path

from datetime import datetime, timezone
RUN_NAME = 'bloom-instruct-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
TINY_SMOKE_TEST = False        # Set True to test locally on CPU in a few seconds
SUBMIT_HF_PR = False           # Explicit opt-in after reviewing results
HF_REPO_ID = 'Centauri-Bloom/Bloom-Text-0.1'
USE_TINYSTORIES = True         # Mix roneneldan/TinyStories (cdla-sharing-1.0) with public-domain-v1
TINYSTORIES_MAX_STORIES = 100_000
TINYSTORIES_VAL_STORIES = 500

STEPS = 300                  # Additional optimizer steps; adjust after evaluation
BATCH_SIZE = 16
GRAD_ACCUM = 2
LEARNING_RATE = 5e-5
WARMUP_STEPS = 50
SAVE_EVERY = 100
EVAL_BATCHES = 32
SEED = 45

if TINY_SMOKE_TEST:
    raise ValueError('Use the original notebook for a fresh CPU smoke test.')

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = Path('/kaggle/working').is_dir()
SAVE_TO_DRIVE = IN_COLAB
if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = Path('/content/drive/MyDrive/centauri-bloom-runs') / RUN_NAME
elif IN_KAGGLE:
    OUT_DIR = Path('/kaggle/working/bloom-runs') / RUN_NAME
else:
    OUT_DIR = Path(os.environ.get('BLOOM_OUTPUT_DIR', str(REPO / 'runs'))) / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=False)

DEVICE = 'cuda' if torch.cuda.is_available() and not TINY_SMOKE_TEST else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('Select an authorized GPU runtime before training.')
print('Device:', DEVICE, '| Output dir:', OUT_DIR)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 3. Restore parent and build a small original instruction corpus
import hashlib, subprocess, random
from tokenizers import Tokenizer
from huggingface_hub import HfApi, snapshot_download
import shutil

PARENT_REVISION = HfApi().model_info(HF_REPO_ID).sha
if not PARENT_REVISION:
    raise RuntimeError('Could not resolve parent revision')
PARENT_DIR = Path(snapshot_download(repo_id=HF_REPO_ID, revision=PARENT_REVISION,
    allow_patterns=['config.json','tokenizer.json','model.safetensors','metrics.json']))
parent_metrics = json.loads((PARENT_DIR / 'metrics.json').read_text())
cfg_raw = json.loads((PARENT_DIR / 'config.json').read_text())
VOCAB_SIZE = cfg_raw['vocab_size']
CONTEXT = cfg_raw['context']
tok_path = OUT_DIR / 'tokenizer.json'
shutil.copyfile(PARENT_DIR / 'tokenizer.json', tok_path)
tokenizer = Tokenizer.from_file(str(tok_path))
if tokenizer.get_vocab_size() != VOCAB_SIZE:
    raise ValueError('Parent tokenizer/config mismatch')
eot_id = tokenizer.token_to_id('<|endoftext|>')
if eot_id is None:
    raise ValueError('Missing original end-of-text token')
print('Parent revision:', PARENT_REVISION, '| original vocab:', VOCAB_SIZE)
print('Reported parent training step:', parent_metrics.get('step'))

pairs=[]
for a in range(1,51):
    for b in range(1,31):
        pairs.extend([(f'What is {a} plus {b}?',str(a+b)),
                      (f'Subtract {b} from {a}.',str(a-b)),
                      (f'Multiply {a} by {b}.',str(a*b))])
pairs.extend([
('What planet do humans live on?','Earth.'),
('What is the nearest star to Earth?','The Sun.'),
('What is the capital of France?','Paris.'),
('What gas do plants use in photosynthesis?','Carbon dioxide.'),
('What is a triangle?','A polygon with three sides.'),
('Define evaporation.','Evaporation is the process by which a liquid becomes a gas at its surface.'),
('Write the word hello in uppercase.','HELLO.'),
('Name three colors.','Red, green and blue.'),
('Say hello politely.','Hello! How can I help?'),
('What is an orbit?','An orbit is the path of an object around another object due to gravity.'),
('What is a computer?','A machine that processes information according to instructions.'),
('Can you be certain about every answer?','No. I can make mistakes and should acknowledge uncertainty.')
])
# Optional: use only a locally reviewed, licensed JSONL file with prompt/response fields.
EXTRA_REVIEWED_JSONL = None
if EXTRA_REVIEWED_JSONL:
    with open(EXTRA_REVIEWED_JSONL,encoding='utf-8') as f:
        for line in f:
            row=json.loads(line)
            if isinstance(row.get('prompt'),str) and isinstance(row.get('response'),str):
                pairs.append((row['prompt'],row['response']))
unique={}
for q,a in pairs:
    q,a=q.strip(),a.strip()
    if q and a:
        unique.setdefault(q,a)
train_pairs,val_pairs=[],[]
for q,a in unique.items():
    digest=hashlib.sha256(q.encode()).hexdigest()
    (val_pairs if int(digest[:8],16)%10==0 else train_pairs).append((q,a))
print('Train/validation pairs:',len(train_pairs),len(val_pairs))
(OUT_DIR/'data-provenance.json').write_text(json.dumps({'recipe':'original arithmetic and basic examples','extra_sha256':hashlib.sha256(Path(EXTRA_REVIEWED_JSONL).read_bytes()).hexdigest() if EXTRA_REVIEWED_JSONL else None,'train_count':len(train_pairs),'validation_count':len(val_pairs)},indent=2))


In [ ]:
# 4. Pack windows and load validated parent weights
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from bloom.model import Bloom, Config
from bloom.checkpoint import config_hash, save_weights, load_weights

def encode_pairs(pairs):
    xs, ys = [], []
    for question, answer in pairs:
        prefix = tokenizer.encode('User: '+question+'\nAssistant:').ids
        response = tokenizer.encode(' '+answer).ids+[eot_id]
        if len(prefix)+len(response)>CONTEXT+1 or not prefix:
            continue
        ids=prefix+response
        x=ids[:-1]
        y=[-100]*(len(prefix)-1)+response
        pad=CONTEXT-len(x)
        xs.append(x+[eot_id]*pad)
        ys.append(y+[-100]*pad)
    if not xs:
        raise RuntimeError('No examples fit context')
    return TensorDataset(torch.tensor(xs,dtype=torch.long),torch.tensor(ys,dtype=torch.long))
train_ds=encode_pairs(train_pairs)
val_ds=encode_pairs(val_pairs)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,drop_last=True)
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False)
if not len(train_loader):
    raise RuntimeError('Too few training examples for batch size')
config = Config(
    vocab_size=tokenizer.get_vocab_size(),
    context=CONTEXT,
    width=cfg_raw['width'],
    heads=cfg_raw['heads'],
    layers=cfg_raw['layers'],
)
torch.manual_seed(SEED)
model = Bloom(config)
if parent_metrics.get('config_sha256') != config_hash(config):
    raise ValueError('Parent configuration hash mismatch')
parent_sha = parent_metrics['weights_sha256']
model.load_state_dict(load_weights(PARENT_DIR / 'model.safetensors', parent_sha, model.state_dict()), strict=True)
model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Restored Bloom: {n_params:,} parameters | config_sha256: {config_hash(config)}')

In [ ]:
# 5. Train with periodic SafeTensors checkpoints saved to Google Drive
import math, time
from itertools import islice
from torch.nn import functional as F

def evaluate(m, loader, device, max_batches):
    m.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for x, y in islice(loader, max_batches):
            x, y = x.to(device), y.to(device)
            with torch.amp.autocast('cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                logits = m(x)
                loss = F.cross_entropy(logits.flatten(0, 1), y.flatten(), reduction='sum')
            total += loss.item()
            count += (y != -100).sum().item()
    return total / count, count

def save_ckpt(step, val_loss, train_loss, elapsed, tokens_seen):
    cpu_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    ckpt_file = OUT_DIR / f'step-{step:06d}.safetensors'
    sha = save_weights(ckpt_file, cpu_state, cpu_state)
    latest_file = OUT_DIR / 'latest.safetensors'
    save_weights(latest_file, cpu_state, cpu_state)
    (OUT_DIR / 'config.json').write_text(json.dumps(config.__dict__, indent=2, sort_keys=True) + '\n')
    meta = {
        'step': int(parent_metrics.get('step', 0)) + step,
        'additional_steps': step,
        'parent_revision': PARENT_REVISION,
        'parent_weights_sha256': parent_sha,
        'parent_reported_train_tokens': parent_metrics.get('train_tokens'),
        'parent_val_loss_same_windows': round(val_before, 4),
        'optimizer_state_restored': False,
        'parameters': n_params,
        'config_sha256': config_hash(config),
        'weights_sha256': sha,
        'train_tokens': tokens_seen,
        'train_tokens_scope': 'this continuation only',
        'last_train_loss': round(train_loss, 4),
        'val_loss': round(val_loss, 4),
        'elapsed_seconds': round(elapsed, 2),
        'use_tinystories': USE_TINYSTORIES,
        'device': DEVICE,
        'run_name': RUN_NAME,
        'dataset': 'original instruction pilot + optional reviewed JSONL',
        'training_track': 'instruction-tuning',
        'data_provenance_sha256': hashlib.sha256((OUT_DIR/'data-provenance.json').read_bytes()).hexdigest(),
        'prompt_format': 'User: {prompt}\\nAssistant:',
        'seed': SEED,
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
        'grad_accum': GRAD_ACCUM,
        'code_revision': subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip(),
    }
    (OUT_DIR / 'metrics.json').write_text(json.dumps(meta, indent=2, sort_keys=True) + '\n')
    # Private run state; never load untrusted optimizer pickle files.
    torch.save({'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
                'step': step, 'tokens_seen': tokens_seen, 'parent_revision': PARENT_REVISION},
               OUT_DIR / 'local-optimizer-state.pt')
    return sha

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1, betas=(0.9, 0.95))
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

def lr_for_step(step):
    if step < WARMUP_STEPS:
        return LEARNING_RATE * (step + 1) / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, STEPS - WARMUP_STEPS)
    return LEARNING_RATE * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))

val_before, val_tokens = evaluate(model, val_loader, DEVICE, EVAL_BATCHES)
print(f'Initial val loss: {val_before:.4f} ({val_tokens} tokens)')

it = iter(train_loader)
tokens_seen = 0
t0 = time.perf_counter()
last_loss = val_before

for step in range(1, STEPS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    for pg in optimizer.param_groups:
        pg['lr'] = lr_for_step(step - 1)
    step_loss = 0.0
    for _ in range(GRAD_ACCUM):
        batch = next(it, None)
        if batch is None:
            it = iter(train_loader)
            batch = next(it)
        x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
        with torch.amp.autocast('cuda', dtype=torch.float16, enabled=(DEVICE == 'cuda')):
            loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten()) / GRAD_ACCUM
        scaler.scale(loss).backward()
        step_loss += loss.item()
        tokens_seen += (y != -100).sum().item()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    last_loss = step_loss

    if step == 1 or step % 100 == 0 or step == STEPS:
        dt = time.perf_counter() - t0
        print(f'step {step:4d}/{STEPS} | loss {last_loss:.4f} | {tokens_seen/max(dt,1e-6):,.0f} tok/s | {dt:.1f}s')
    if step % SAVE_EVERY == 0 or step == STEPS:
        vloss, _ = evaluate(model, val_loader, DEVICE, EVAL_BATCHES)
        sha = save_ckpt(step, vloss, last_loss, time.perf_counter() - t0, tokens_seen)
        print(f'  -> Saved checkpoint step {step} | val_loss {vloss:.4f} | sha256 {sha[:12]}...')

val_after, _ = evaluate(model, val_loader, DEVICE, EVAL_BATCHES)
print(f'Done in {time.perf_counter() - t0:.1f}s | Val loss: {val_before:.4f} -> {val_after:.4f}')

In [ ]:
# 6. Generate sample completions
def generate(prompt, max_new_tokens=128, temperature=0.8, top_k=40):
    model.eval()
    ids = tokenizer.encode(prompt).ids or [eot_id]
    idx = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            ctx = idx[:, -config.context:]
            logits = model(ctx)[:, -1, :] / max(temperature, 1e-5)
            if top_k:
                vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < vals[:, [-1]]] = float('-inf')
            probs = torch.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            if nxt.item() == eot_id:
                break
            idx = torch.cat([idx, nxt], dim=1)
    return tokenizer.decode(idx[0].tolist())

for p in ['User: What is 17 plus 23?\nAssistant:', 'User: What is a star?\nAssistant:', 'User: Name three planets.\nAssistant:']:
    print('=== PROMPT:', repr(p), '===')
    print(generate(p))
    print()

In [ ]:
# 7. Review separately; do not overwrite the base model
print((OUT_DIR/'metrics.json').read_text())
print('Candidate:',OUT_DIR)
print('Evaluate held-out instruction prompts and general completion regression before proposing a separate instruct checkpoint.')
